In [1]:
import wandb
from wandb.integration.keras import WandbMetricsLogger

import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras

In [2]:
sweep_config = {

    'method'   : 'grid',
    'metric'   : {
        'name' : 'val_accuracy',
        'goal' : 'maximize'
                 },
    'parameters' : {
        'batch_size'    : {'values': [8,16]},
        'learning_rate' : {'values': [0.001,0.0001]},
        'hidden_nodes'  : {'values': [128,64]},
        'img_size'      : {'values': [16, 224]},
        'epochs'        : {'values': [5,10]}
    }
}

sweep_id = wandb.sweep(sweep_config, project="5-flowers-classification")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Create sweep with ID: 472kegn8
Sweep URL: https://wandb.ai/jaicky-iit-ism-dhanbad/5-flowers-classification/sweeps/472kegn8


In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kausthubkannan/5-flower-types-classification-dataset")

print("Path to dataset files:", path)

100%|██████████| 242M/242M [00:01<00:00, 182MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/kausthubkannan/5-flower-types-classification-dataset/versions/1


In [4]:
import os
print(os.listdir(path))
new_path = os.path.join(path, "flower_images")
print(os.listdir(new_path))

['flower_images']
['Tulip', 'Lilly', 'Orchid', 'Lotus', 'Sunflower']


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

image_paths = []
labels = []

# Define CLASS_NAMES here so it's accessible
CLASS_NAMES = ['Lilly', 'Tulip', 'Sunflower', 'Lotus', 'Orchid']

# 'new_path' is already defined from cell ODzq1vCHVNVj
# 'CLASS_NAMES' is already defined from cell WFuC9l3YS4vl

# Populate image_paths and labels
for class_name in CLASS_NAMES:
    class_dir = os.path.join(new_path, class_name)
    if os.path.exists(class_dir):
        for img_name in os.listdir(class_dir):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_paths.append(os.path.join(class_dir, img_name))
                labels.append(class_name)

if not image_paths:
    print("No image files found. Please verify the dataset structure and 'new_path'.")
else:
    df = pd.DataFrame({'filename': image_paths, 'label': labels})

    # Split into training and evaluation sets
    train_df, eval_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

    # Save to CSV files in the current working directory
    train_csv_path = 'train_set.csv'
    eval_csv_path = 'eval_set.csv'
    train_df.to_csv(train_csv_path, index=False, header=False)
    eval_df.to_csv(eval_csv_path, index=False, header=False)

    print(f"Generated training CSV: {train_csv_path} ({len(train_df)} samples)")
    print(f"Generated evaluation CSV: {eval_csv_path} ({len(eval_df)} samples)")

    print("\nTrain DataFrame Head:")
    display(train_df.head())
    print("\nEval DataFrame Head:")
    display(eval_df.head())

Generated training CSV: train_set.csv (3999 samples)
Generated evaluation CSV: eval_set.csv (1000 samples)

Train DataFrame Head:


,filename,label
3173,/root/.cache/kagglehub/datasets/kausthubkannan...,Lotus
1215,/root/.cache/kagglehub/datasets/kausthubkannan...,Tulip
1914,/root/.cache/kagglehub/datasets/kausthubkannan...,Tulip
2417,/root/.cache/kagglehub/datasets/kausthubkannan...,Sunflower
193,/root/.cache/kagglehub/datasets/kausthubkannan...,Lilly



Eval DataFrame Head:


,filename,label
3446,/root/.cache/kagglehub/datasets/kausthubkannan...,Lotus
2502,/root/.cache/kagglehub/datasets/kausthubkannan...,Sunflower
955,/root/.cache/kagglehub/datasets/kausthubkannan...,Lilly
4880,/root/.cache/kagglehub/datasets/kausthubkannan...,Orchid
4617,/root/.cache/kagglehub/datasets/kausthubkannan...,Orchid


In [6]:
def train():
  with wandb.init() as run:
    config = wandb.config

    # Constants
    IMG_HEIGHT = config.img_size
    IMG_WIDTH = config.img_size
    IMG_CHANNELS = 3
    CLASS_NAMES = ['Lilly', 'Tulip', 'Sunflower', 'Lotus', 'Orchid']



    def read_and_decode(filename, resize_dims):
        img_bytes = tf.io.read_file(filename)
        img = tf.image.decode_jpeg(img_bytes, channels=IMG_CHANNELS)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.resize(img, resize_dims)
        return img

    def parse_csvline(csv_line):
        record_default = ["", ""]
        filename, label_string = tf.io.decode_csv(csv_line, record_default)

        img = read_and_decode(filename, [IMG_HEIGHT, IMG_WIDTH])

        # Convert label string to an integer index
        label = tf.where(tf.equal(CLASS_NAMES, label_string))[0, 0]

        return img, label

    # Define datasets
    train_dataset = (
        tf.data.TextLineDataset(os.path.join(os.getcwd(), train_csv_path))
        .skip(1) # Skip header if present, but we generated without header
        .map(parse_csvline, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(config.batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )

    eval_dataset = (
        tf.data.TextLineDataset(os.path.join(os.getcwd(), eval_csv_path))
        .skip(1) # Skip header if present, but we generated without header
        .map(parse_csvline, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(config.batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )


    #architecture
    model = keras.Sequential([
        keras.layers.Flatten(input_shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)),
        keras.layers.Dense(config.hidden_nodes, activation="relu"),
        keras.layers.Dense(len(CLASS_NAMES), activation="softmax")
    ])

    model.compile(
          optimizer=keras.optimizers.Adam(learning_rate=config.learning_rate),
          loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
          metrics=["accuracy"]
      )

    model.fit(
      train_dataset,
      validation_data=eval_dataset,
      epochs=config.epochs,
      callbacks=[WandbMetricsLogger(log_freq=5)]
      )

In [7]:
wandb.agent(sweep_id, function=train)

wandb: Agent Starting Run: nsku5rpx with config:
wandb: 	batch_size: 8
wandb: 	epochs: 5
wandb: 	hidden_nodes: 128
wandb: 	img_size: 16
wandb: 	learning_rate: 0.001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: jaikeeanand007 (jaicky-iit-ism-dhanbad) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
    500/Unknown 13s 23ms/step - accuracy: 0.3537 - loss: 1.5352

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


500/500 ━━━━━━━━━━━━━━━━━━━━ 17s 31ms/step - accuracy: 0.4082 - loss: 1.4218 - val_accuracy: 0.4655 - val_loss: 1.3067
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 19s 37ms/step - accuracy: 0.4817 - loss: 1.2496 - val_accuracy: 0.4735 - val_loss: 1.2409
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 32s 64ms/step - accuracy: 0.5105 - loss: 1.1989 - val_accuracy: 0.5105 - val_loss: 1.2003
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 14s 28ms/step - accuracy: 0.5318 - loss: 1.1367 - val_accuracy: 0.5105 - val_loss: 1.1764
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 14s 28ms/step - accuracy: 0.5625 - loss: 1.0893 - val_accuracy: 0.5055 - val_loss: 1.1712


...

batch/accuracy,▁▂▃▄▅▅▅▆▆▆█▇██▇█▇▇▇▆▇▇▇▁▁▃▃▃▂▂▃▃▂▂▂▂▃▃▃▃
batch/batch_step,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▆▆▆▆▆▆▆▆▆▆▆▇▇▇████
batch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,█▇▆█▅▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,▄▇█▆▆▆▂▁▁▂
epoch/epoch,▁▂▃▃▄▅▆▆▇█
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▄▂▁▁▁▁▁▁▁
epoch/val_accuracy,▅█▃▅▆▁▄▂▄▃
epoch/val_loss,█▂▅▁▁▁▁▁▁▁
batch/accuracy,0.2876


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: 5ik1ee67 with config:
wandb: 	batch_size: 16
wandb: 	epochs: 10
wandb: 	hidden_nodes: 64
wandb: 	img_size: 224
wandb: 	learning_rate: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 18s 66ms/step - accuracy: 0.3927 - loss: 1.9291 - val_accuracy: 0.4645 - val_loss: 1.4144
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 16s 63ms/step - accuracy: 0.4572 - loss: 1.6278 - val_accuracy: 0.4174 - val_loss: 2.1344
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 22s 86ms/step - accuracy: 0.5053 - loss: 1.4713 - val_accuracy: 0.4024 - val_loss: 2.3361
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 16s 62ms/step - accuracy: 0.5248 - loss: 1.3779 - val_accuracy: 0.4384 - val_loss: 2.1024
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 16s 62ms/step - accuracy: 0.5623 - loss: 1.3097 - val_accuracy: 0.4575 - val_loss: 1.8553
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 16s 62ms/step - accuracy: 0.5688 - loss: 1.2449 - val_accuracy: 0.4555 - val_loss: 1.7430
Epoch 7/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 82s 329ms/step - accuracy: 0.5953 - loss: 1.1873 - val_accuracy: 0.4975 - val_loss: 1.7126
Epoch 8/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 16s 62ms/step - accuracy: 0.6088 - loss: 1.1258 -

batch/accuracy,▁▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▄▇▆▆▇▇▇▇▇▇▇▇▇▇▇████
batch/batch_step,▁▁▁▁▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇█████
batch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,█▆▂▂▂▂▂▂▂▂▂▂▂▂▁▂▂▁▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,▁▃▄▅▆▆▇▇██
epoch/epoch,▁▂▃▃▄▅▆▆▇█
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▅▄▃▃▂▂▁▁▁
epoch/val_accuracy,▅▂▁▃▅▅██▄▄
epoch/val_loss,▁▆█▆▄▃▃▃█▇
batch/accuracy,0.63364


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.
